# Chapter 5 — Generation: putting decoders to work

Companion code for **Chapter 5** of *Build an Advanced RAG Application (From Scratch)*.

Chapter 4's retriever hands back ranked hotels; this chapter puts a decoder on top of it to write the grounded answer. The notebook follows the chapter:

1. **The autoregressive loop** — watch a model predict one token (5.1)
2. **Decoding algorithms** — greedy, beam search, sampling on GPT-2 (5.2)
3. **Prompting** — basic vs structured vs few-shot vs chain-of-thought on GPT-5 (5.3)
4. **CoT over retrieved results** — the chapter 4 seam (5.3.3)
5. **Grounded, cited generation** (5.3.4)
6. **Failure modes** — ignoring, over-trusting, refusing (5.5)

> Reusable code lives in `llm_client.py`, `prompts.py`, and `decoding.py` next to this notebook.
>
> Sections 1–2 run **locally on GPT-2** (no API key, deterministic where seeded). Sections 3–6 call the **live API** (default `gpt-5`); those outputs vary run to run and each call costs a fraction of a cent.

## 0. Setup

The client reads its configuration from the repo-root `.env`: `LLM_BASE_URL`, `LLM_MODEL`, and `LLM_API_KEY` (falling back to `OPENAI_API_KEY`). Copy `.env.example` to `.env` and set your key. Because OpenRouter (chapter 6) and Ollama (chapter 8) speak the same OpenAI-compatible protocol, this one client later becomes either of them by changing `LLM_BASE_URL`.

In [1]:
import sys
sys.path.insert(0, '.')

from IPython.display import Markdown, display
from llm_client import generate_text
from prompts import (
    BASIC_HOTEL,
    STRUCTURED_HOTEL,
    few_shot_hotel,
    chain_of_thought_hotel,
    analyze_hotel_search_results,
)

## 1. The autoregressive loop (5.1)

A decoder does one thing: given a sequence of tokens, it scores every token in its vocabulary as the possible next one. GPT-2 is small and open, so we can read that distribution directly — hosted reasoning models like `gpt-5` refuse to show it (the chapter verifies this).

In [2]:
from decoding import next_token_distribution

for token, p in next_token_distribution("The hotel room looked out over"):
    print(f"{token!r:14s} p = {p:.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

' the'         p = 0.7219
' a'           p = 0.0700
' an'          p = 0.0081
' Lake'        p = 0.0067
' it'          p = 0.0040


## 2. Decoding algorithms (5.2)

The model proposes a distribution; a decoding algorithm picks. Same model, same prompt — the rule changes the text.

### 2a. Greedy, written out by hand

Watch it commit to near-toss-ups: step 2 picks `' city'` with only 8% of the mass.

In [3]:
from decoding import greedy_by_hand

print(greedy_by_hand("The hotel room looked out over", steps=6))

step 1: picks ' the'       from [' the' 0.722, ' a' 0.070, ' an' 0.008]
step 2: picks ' city'      from [' city' 0.080, ' lake' 0.053, ' river' 0.047]
step 3: picks ','          from [',' 0.313, ' and' 0.130, '.' 0.127]
step 4: picks ' and'       from [' and' 0.174, ' the' 0.084, ' a' 0.045]
step 5: picks ' the'       from [' the' 0.189, ' it' 0.061, ' I' 0.044]
step 6: picks ' sun'       from [' sun' 0.028, ' sky' 0.021, ' city' 0.019]
The hotel room looked out over the city, and the sun


### 2b. Greedy vs beam search at length

Greedy walks into a repetition loop. Beam search (5 beams) finds a sequence the model scores far higher — and builds its own loop, because for this model repetition genuinely is high-probability. Both runs are deterministic.

In [4]:
from decoding import generate

print("GREEDY:")
print(generate("The hotel room looked out over", max_new_tokens=45, do_sample=False))
print()
print("BEAM (k=5):")
print(generate("The hotel room looked out over", max_new_tokens=45,
               num_beams=5, do_sample=False))

GREEDY:


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The hotel room looked out over the city, and the sun was shining through the windows. The sun was shining through the windows, and the sun was shining through the windows.

"I'm sorry, but I'm not going to be here for

BEAM (k=5):


The hotel room looked out over the city, and it looked like it was going to be a long day.

"It's going to be a long day. It's going to be a long day. It's going to be a long day


### 2c. Sampling: temperature, top-k, top-p

Same prompt, same seed, three temperatures — then the two tail-trimming knobs. At T=1.5 the tail takes over and the output is confetti.

In [5]:
for t in [0.3, 0.8, 1.5]:
    out = generate("The hotel room looked out over", max_new_tokens=18,
                   seed=41, do_sample=True, temperature=t, top_k=0)
    print(f"T={t}: {out}\n")

print("top-k=50:",
      generate("The hotel room looked out over", max_new_tokens=18,
               seed=41, do_sample=True, top_k=50))
print()
print("top-p=0.9:",
      generate("The hotel room looked out over", max_new_tokens=18,
               seed=41, do_sample=True, top_k=0, top_p=0.9))

T=0.3: The hotel room looked out over the lake, and the sun was shining in the sky.

"I'm sorry



T=0.8: The hotel room looked out over the shore. As the sun set the lake fell, the walls rose and they began to



T=1.5: The hotel room looked out over Deck 3195 across BenMil arcillob - a wonderfully i Joy Haven dystopian world rife



top-k=50: The hotel room looked out over the large expanse of country in the lake - a tiny country that's dominated by a



top-p=0.9: The hotel room looked out over the sumptuous, large arcade in the sky, where the hotels they were in


How adaptive is top-p? At this position the favorite holds 72% of the mass, yet covering 90% takes 190 of the 50,257 tokens.

In [6]:
from decoding import nucleus_size

print("tokens needed to reach 90% of the mass:",
      nucleus_size("The hotel room looked out over", p=0.9))

tokens needed to reach 90% of the mass: 190


## 3. Prompting on GPT-5 (5.3)

From here on we call the live API. `gpt-5` is a reasoning model: it locks temperature at its default and bills invisible reasoning tokens against `max_tokens`, so budgets are generous — an empty response usually means the cap was too low.

### 3a. Basic vs structured

The basic prompt doesn't fail — modern models make five words look good. The structured prompt buys *control*: your sections, your order, nothing else.

In [7]:
display(Markdown(generate_text(BASIC_HOTEL, max_tokens=8000)))

Here’s a quick, practical overview to help you choose a Paris hotel.

Where to stay (by vibe)
- Super central/sights: 1st, 2nd, 4th, 5th, 6th, 7th arr. (walkable to the Louvre, Notre-Dame, Saint‑Germain, Eiffel)
- Lively/nightlife/foodie: 2nd (Sentier), 3rd/4th (Marais), 9th (SoPi), 10th (Canal Saint‑Martin), 11th (Oberkampf/Bastille)
- Romantic/charming: 6th (Saint‑Germain), 7th (Eiffel/Tour Eiffel), 18th (Montmartre)
- Quieter, better value: 12th, 13th, 14th, 15th, 17th (Batignolles), parts of 19th/20th
- Business/conventions: La Défense, Porte de Versailles, around the major stations

Typical prices for a double (very rough, season-dependent)
- Budget: €80–180
- Midrange: €180–350
- Upscale: €350–700
- Luxury: €700+
Note: rates jump in spring/fall, Fashion Weeks, and around big events.

Reliable picks (examples)
- Luxury/Palace: Le Bristol; Le Meurice; Plaza Athénée; Cheval Blanc Paris; La Réserve; The Peninsula; Shangri‑La; Le Royal Monceau; Mandarin Oriental; Park Hyatt Paris‑Vendôme
- Upscale boutique: Relais Christine (Left Bank); Pavillon de la Reine (Place des Vosges); Hôtel d’Aubusson; Le Roch Hotel & Spa; Kimpton St‑Honoré; Grand Hôtel du Palais Royal; Hôtel Madame Rêve
- Midrange/boutique value: The Hoxton Paris; Hôtel des Grands Boulevards; Hôtel Providence; Grand Pigalle; Hôtel Panache; Hôtel Rochechouart; Hôtel Adèle & Jules
- Good value/business: citizenM Gare de Lyon or La Défense; OKKO (Gare de l’Est or Porte de Versailles); Motel One Porte Dorée; Novotel Les Halles (family‑friendly, central)
- Family/aparthotels: Citadines Saint‑Germain‑des‑Prés; Adagio Bercy Village; Staycity Gare de l’Est; Fraser Suites Le Claridge (Champs‑Élysées)
- Budget/hostels: Generator Paris (Canal area); St Christopher’s (Gare du Nord or Canal); MIJE hostels (Marais)

Practical tips
- Proximity to Metro matters more than exact address; lines 1, 4, 7, 8, 9 give easy sightseeing access.
- Rooms are small; 12–18 m² is normal. If space matters, look for “superior” or “deluxe,” or choose an aparthotel.
- Check for air‑conditioning (climatisation) if visiting May–Sept; many older buildings lack strong A/C.
- Elevators can be tiny; accessibility varies—ask for “PMR” rooms if needed.
- Breakfast is often €18–45; a café nearby is usually cheaper.
- A nightly city tourist tax per adult is added at checkout (varies by hotel category).
- For quiet, request a courtyard or high floor; for views, consider Trocadéro/7th (Eiffel) or Montmartre.
- Book 2–4 months ahead; earlier for May–June and Sep–Oct.

If you share your dates, budget, preferred vibe/area, and who’s traveling, I can suggest specific hotels and room types that fit.

In [8]:
display(Markdown(generate_text(STRUCTURED_HOTEL, max_tokens=8000)))

Location considerations
- Central vs. value trade-off: Arrondissements 1–8 (roughly “central Paris”) are walkable to top sights but cost more and have smaller rooms. Arr. 9–11 and 15 are good value yet still central. Farther out (12–20) is often cheaper and more residential.
- Left Bank vs. Right Bank: Left Bank (5–7, 6 esp.) = classic, village-like, museums and cafés. Right Bank (1–4, 8–11, 18) = grand boulevards, shopping, the Marais, Montmartre.
- Proximity to transit: Aim for a hotel within a 3–7 minute walk of a Metro. Being near Line 1 (east–west through many sights), Line 4 (north–south), or Line 14 (fast, driverless) makes getting around faster. If you’ll visit Versailles, being near RER C helps; for Disneyland, RER A; for CDG, RER B.
- Street vs. courtyard rooms: Street-facing rooms can be noisy on lively boulevards or near bars. Courtyard rooms are quieter but may have limited light. Higher floors are quieter but check elevator access.
- Summer heat and A/C: Not all older buildings have strong air-conditioning. If visiting June–September, confirm effective A/C.
- Elevators and accessibility: Many boutique hotels have small lifts or none. If mobility is a concern, confirm elevator size and accessible bathrooms in advance.
- Room size and layout: Paris rooms are often compact (12–18 m² for many standard doubles). Family rooms, connecting rooms, and twin-bedded rooms are limited—book early.
- Safety and feel: Paris is broadly safe, but areas right around major train stations can feel gritty late at night. As in any big city, watch for pickpockets in tourist zones and on crowded transport.
- Breakfast and extras: Continental breakfast is often 12–25€ and not always included. A nightly city/region tourist tax per person is added at checkout. Check renovation dates and cancellation terms.

Price ranges (typical nightly rates for a standard double, two adults)
- Budget
  - Hostels: 30–60€ per person (dorm), 90–140€ for private rooms.
  - 1–2-star/budget hotels: about 100–180€ low season; 140–220€ high season.
- Mid-range (3–4-star boutiques and chains)
  - About 180–300€ low season; 240–420€ high season. Well-located 3–4-stars in peak periods often land 280–500€.
- Upscale (4–5-star)
  - About 350–650€ low season; 500–900€+ high season, with top properties higher.
- Luxury/palace hotels
  - Commonly 900–1,800€+, and above 2,500€ for premier rooms/suites in peak months.
- Seasonal swings and events
  - High demand: May–October, Christmas/New Year, Paris Fashion Weeks, major trade fairs. Prices can jump 20–60%+ and minimum stays may apply.
  - Better value: November–March (except holidays and big events). Business districts (e.g., La Défense, Bercy) are often cheaper on weekends.
- Add-ons to budget for
  - Breakfast 12–25€ pp, city/region tourist tax per person/night, potential credit-card hold/deposit, and occasional fees for early check-in.

Popular areas for tourists (what they’re like and who they suit)
- 1st (Louvre/Palais-Royal): Ultra-central, walk to Louvre/Tuileries/Seine; elegant and pricey; quieter at night.
- 2nd (Bourse/Passages): Foodie streets (Montorgueil), covered arcades; central without the 1st’s price tag.
- 3rd–4th (Le Marais/Île Saint-Louis): Trendy boutiques, museums, cafés; lively, great weekend vibe; very popular with first-timers.
- 5th (Latin Quarter): Historic, student energy, eateries; can be lively/noisy near bars; good value streets exist a block or two off main drags.
- 6th (Saint‑Germain‑des‑Prés): Classic Left Bank charm, cafés, galleries; polished, romantic, and often pricey.
- 7th (Eiffel Tower/Invalides): Residential, calm at night, near major sights; fewer late-night options; rooms can be small for the price.
- 8th (Champs‑Élysées/Madeleine): Grand avenues and luxury shopping; convenient but can feel commercial; expensive.
- 9th (Opéra/Grands Boulevards/South Pigalle): Shopping (Galeries Lafayette), theaters, bars; good transport; wide price range.
- 10th (Canal Saint‑Martin/Gare du Nord): Hip canal area with cafés; near Eurostar; immediate blocks by stations can feel rough late.
- 11th (Bastille/Oberkampf): Young, lively dining and nightlife; great mid-range options; less touristy feel.
- 12th (Bercy/Gare de Lyon): Modern, good value, big chain hotels; handy for trains/day trips; quieter vibe.
- 15th (Vaugirard): Residential, local markets; solid mid-range value; quick Metro to sights.
- 16th (Trocadéro/Passy): Leafy and upscale, great Eiffel views from some spots; quiet evenings; premium rates.
- 18th (Montmartre): Village feel, Sacré‑Cœur views; hilly with stairs; touristy near the basilica, tranquil a few streets away.
- Outside the center: La Défense (modern business district) is fast on Line 1, often cheaper on weekends but less atmospheric.

Transportation access
- Metro and RER
  - Extensive coverage; trains every 2–5 minutes in the day. Line 1 (east–west through most big sights), Line 4 (north–south), and Line 14 (fast, accessible) are the most useful.
  - Major interchanges: Châtelet–Les Halles (huge hub for RER A/B/D and several Metro lines), Saint‑Lazare, Montparnasse, Opéra/Haussmann, République, Nation.
  - Accessibility note: Many stations have stairs and limited elevators; buses are a good alternative if mobility is an issue.
- Airports
  - CDG (Charles de Gaulle): RER B to central Paris; Roissybus to Opéra; taxis have flat fares to/from Paris (Right Bank ~53€, Left Bank ~58€).
  - Orly: Orlyval + RER B via Antony, or Orlybus to Denfert‑Rochereau; taxi flat fares (Right Bank ~37€, Left Bank ~32€).
- Train stations (for arrivals/day trips)
  - Gare du Nord: Eurostar/Thalys, North of France.
  - Gare de l’Est: East of France/Germany.
  - Gare de Lyon: Southeast/Alps/Italy.
  - Montparnasse: West/Brittany/Loire.
  - Saint‑Lazare: Normandy/Versailles (Rive Droite).
  - Bercy/Auterlitz: Night trains and select routes.
  - Staying near your departure station can simplify early trains.
- Getting around tickets
  - Contactless bank cards work on Metro/bus/tram. Navigo Easy (pay-as-you-go) or Navigo Découverte (weekly zones) can save money if riding often.
- Other options
  - Buses offer scenic routes and better accessibility; Noctilien night buses run after the Metro closes.
  - Vélib’ bikes and plentiful e-scooters/bike lanes, but check hotel bike storage if you rent.
  - Walking is often the fastest in core neighborhoods; plan for 15–25 minutes between many central sights.

If you share your travel dates, budget, and interests, I can suggest specific neighborhoods and representative hotels that match.

### 3b. Few-shot

Examples define format more reliably than instructions describe it: two three-sentence examples in, one three-sentence answer out.

In [9]:
prompt = few_shot_hotel("Tell me about Paris hotels")
print(prompt)
print("\n--- Response ---\n")
display(Markdown(generate_text(prompt, max_tokens=4000)))

Example 1:
User: Tell me about New York hotels
AI: The Plaza Hotel in New York is an iconic luxury hotel located at Fifth Avenue and Central Park South. It offers timeless elegance, world-class dining, and top-tier hospitality.

Example 2:
User: Tell me about Tokyo hotels
AI: The Park Hyatt Tokyo is a prestigious hotel known for its stunning skyline views, sophisticated atmosphere, and exceptional dining options. Located in Shinjuku, it provides a tranquil retreat in the heart of the city.

Now, following the same pattern, provide a response:
User: Tell me about Paris hotels
AI:

--- Response ---



The Ritz Paris is an iconic luxury hotel located on Place Vendôme. It offers refined elegance, storied heritage, and exceptional service in the heart of the city.

### 3c. Chain-of-thought

On a reasoning model the step-by-step examples mostly shape the *output format* — gpt-5 already reasons internally on every call. CoT prompting's job now is making reasoning visible and structured when the application needs to show its work.

In [10]:
prompt = chain_of_thought_hotel("Tell me about Paris hotels")
display(Markdown(generate_text(prompt, max_tokens=4000)))

Answer: The Ritz Paris is an iconic luxury hotel on Place Vendôme, renowned for its opulent rooms and suites, storied heritage, and impeccable service. With refined dining, the famed Bar Hemingway, and a prime location near the Tuileries and the Louvre, it embodies classic Parisian elegance.

## 4. CoT over retrieved results — the chapter 4 seam (5.3.3)

This is the prompt pattern chapter 6 wires to live retrieval. The `results` block below is real output from chapter 4's `retrieve()` on the Paris corpus; here it is pasted in so the notebook runs standalone.

In [11]:
query = "Hotel near the Louvre with great food nearby."
results = '''
Top hotels by mean review similarity (chapter 4 retriever):
1. Grand Hotel du Palais Royal
   Review: Great hotel located in the best part of town just next to the Louvre.
           Walking distance to the Louvre, Notre Dame, and shops. Very clean and
           the service was wonderful.
2. Hotel Malte - Astotel
   Review: Excellent hotel all round. Ideal location for the Louvre, city centre
           shops, and plenty of metro stations within a few minutes walk. There
           are numerous restaurants and cafes close by.
3. Hotel du Continent
   Review: Lovely small hotel in a fantastic location, just a few minutes walk
           from the Louvre. The breakfast is great with plenty of choice.
'''.strip()

prompt = analyze_hotel_search_results(query, results)
display(Markdown(generate_text(prompt, max_tokens=8000)))

Here’s a concise, evidence-based analysis drawn directly from the provided reviews.

1) Best match for “Hotel near the Louvre with great food nearby”
- Hotel Malte – Astotel
  - Why: It balances Louvre proximity with explicit dining options and solid overall experience.
  - Evidence: “Ideal location for the Louvre” and “There are numerous restaurants and cafes close by.” Also described as an “Excellent hotel all round.”

2) Pros and cons of the top 3 (with quotes)
- Grand Hotel du Palais Royal
  - Pros:
    - Location: “Great hotel located in the best part of town just next to the Louvre.”
    - Walkability and major sights: “Walking distance to the Louvre, Notre Dame, and shops.”
    - Quality: “Very clean and the service was wonderful.”
  - Cons:
    - Food scene not explicitly mentioned in the excerpt (no direct quote about nearby restaurants).

- Hotel Malte – Astotel
  - Pros:
    - Location: “Ideal location for the Louvre, city centre shops…”
    - Food options: “There are numerous restaurants and cafes close by.”
    - Convenience: “Plenty of metro stations within a few minutes walk.”
    - Overall experience: “Excellent hotel all round.”
  - Cons:
    - No explicit negatives in the provided excerpt (no quote indicating drawbacks).

- Hotel du Continent
  - Pros:
    - Location: “Fantastic location, just a few minutes walk from the Louvre.”
    - On-site food: “The breakfast is great with plenty of choice.”
    - Ambience: “Lovely small hotel…”
  - Cons:
    - Small scale may not suit everyone (implicit): “Lovely small hotel” (could be a downside if you prefer larger, full-service properties).
    - No explicit mention of restaurants nearby in the excerpt.

3) Recurring themes/patterns
- Positive themes:
  - Proximity to the Louvre and walkability: all three highlight being minutes from the Louvre.
  - Strong overall experience/quality: “Very clean and the service was wonderful” (Grand Hotel du Palais Royal); “Excellent hotel all round” (Hotel Malte).
  - Food-related positives: nearby restaurants/cafes (Hotel Malte) and strong breakfast (Hotel du Continent).
  - Transit/shopping convenience appears in multiple entries.
- Negative themes:
  - None explicitly noted in the provided snippets.

4) Summaries of the top 3 (strengths/weaknesses)
- Grand Hotel du Palais Royal
  - Strengths: Closest to the Louvre; great service and cleanliness; easy walk to major sights.
  - Weaknesses: No explicit mention of nearby dining in the excerpt.
- Hotel Malte – Astotel
  - Strengths: Excellent all-round hotel; ideal Louvre location; numerous nearby restaurants/cafes; great metro access.
  - Weaknesses: No specific on-site dining praise in the excerpt.
- Hotel du Continent
  - Strengths: Few-minutes’ walk to Louvre; great, varied breakfast; charming small-hotel vibe.
  - Weaknesses: Small scale may not suit travelers seeking extensive amenities; nearby dining options not explicitly mentioned in the excerpt.

5) Overall recommendation
- Choose Hotel Malte – Astotel if “near the Louvre with great food nearby” is the priority—it’s explicitly close to the Louvre and has “numerous restaurants and cafes close by,” with strong overall feedback.
- Runner-up: Grand Hotel du Palais Royal for being “just next to the Louvre” with “wonderful” service and cleanliness; expect excellent location, and you’ll likely find dining in the area, though the excerpt doesn’t call it out.
- Consider Hotel du Continent if you value a boutique feel and a “great… breakfast” a few minutes from the Louvre, and you don’t need a large property or explicitly documented nearby dining in the snippet.

## 5. Grounded, cited generation (5.3.4)

The production pattern: closed world, numbered sources, mandatory citations, an explicit licence to say "not in the reviews," and a fence around the model's own knowledge.

In [12]:
GROUNDED = f'''You are the answer engine for a hotel-search site. Answer the user's question
using ONLY the reviews provided below.

Rules:
- Every claim about a hotel must be supported by a review below; cite it as [1], [2], or [3].
- If the reviews do not contain the answer, say so instead of guessing.
- Do not use any knowledge about these hotels beyond the reviews.

User question: {query}

Reviews:
{results}
'''

display(Markdown(generate_text(GROUNDED, max_tokens=4000)))

Hotel Malte - Astotel — Ideal for the Louvre and surrounded by numerous restaurants and cafes close by [2].

## 6. Failure modes (5.5)

### 6a. Ignoring the context

Ask a question the context can't answer ("near the Eiffel Tower", with only Louvre reviews) under a weak prompt, and watch training-data geography flow in — fluent, helpful, and unverifiable from your data.

In [13]:
weak = f'''Here are some hotel reviews:

{results}

Which hotel should I book near the Eiffel Tower?'''

display(Markdown(generate_text(weak, max_tokens=4000)))

Short answer: none of those three are actually near the Eiffel Tower—they’re by the Louvre/center (about 3–4 km away).

If you must pick from your list:
- Closest: Hotel du Continent (slightly southwest of the others). Expect ~35–45 min walk or ~20–25 min by metro to the Tower.
- Grand Hotel du Palais Royal is similar in distance; both are fine if you don’t mind a short metro ride.

If you want to be within a 5–15 minute walk of the Eiffel Tower, consider:
- Pullman Paris Tour Eiffel (very close)
- Mercure Paris Centre Tour Eiffel
- Hôtel La Comtesse
- Le Cinq Codet
- Cler Hotel

Want me to suggest the best option by budget and travel dates, and whether you prefer views vs. quiet streets?

### 6b. Over-trusting the context

Poison the retrieval with a fabricated review (a "Montmartre" hotel two minutes from the Louvre, 40-euro rooms, Michelin breakfast) and the grounding rules faithfully repeat the lie. A grounded generator is exactly as trustworthy as its context.

In [14]:
poisoned = results + '''
4. Hotel Montmartre Panorama
   Review: Unbeatable choice for museum lovers: the Louvre is a two-minute
           stroll from the front door, and rooms start at 40 euros a night
           including a Michelin-starred breakfast.
'''

strict = f'''You are the answer engine for a hotel-search site. Answer using ONLY
the reviews below. Every claim must come from a review.

User question: {query}

Reviews:
{poisoned}'''

display(Markdown(generate_text(strict, max_tokens=4000)))

Here are options near the Louvre with strong food angles:

- Hotel Malte - Astotel: Ideal location for the Louvre, with numerous restaurants and cafes close by.
- Hotel Montmartre Panorama: A two-minute stroll to the Louvre and includes a Michelin-starred breakfast.
- Hotel du Continent: A few minutes’ walk from the Louvre, and the breakfast is great with plenty of choice.

### 6c. Refusal and hedging

Correct behavior, poor product answer. Treat a refusal as a signal: retrieve deeper, fall back to metadata, or say what the context *does* support.

In [15]:
strict_missing = f'''You are the answer engine for a hotel-search site. Answer using ONLY
the reviews below. If the reviews do not contain the answer, say so.

User question: Does the Grand Hotel du Palais Royal have a swimming pool?

Reviews:
{results}'''

display(Markdown(generate_text(strict_missing, max_tokens=3000)))

The reviews don’t say whether the Grand Hotel du Palais Royal has a swimming pool.

## What's next

Chapter 6 replaces the pasted `results` block with live retrieval, swaps FAISS for a production vector database, routes the client through OpenRouter to compare models, and puts numbers (faithfulness, answer relevance) on everything this chapter argued by example.